# Stage 3: Compare the Checkpoints

Here I evaluate the checkpoints I have and generate text from the same prompts. The scores go into `results/model_comparison_results.csv`; the generated text goes into `results/model_comparison_prompt_results.csv`.

The notebook skips checkpoints that are missing. So far, I only have results for the 3M model. The generated text is a random sample, not a formal quality score.


In [11]:
from pathlib import Path
import pandas as pd
import torch
from tokenizers import Tokenizer
from create_model import create_model
import numpy as np
import math
import gc

In [12]:
prompts = [
    "Столиця України",
    "Президент",
    "Одного вечора",
    "Найбільша річка в Україні",
    "Ukraine is"
]
MODEL_DIR = Path("models")
DATA_TEST_DIR = Path("data/test")
MODELS = ["3M", "9M", "20M"]
DATA_TOKENS = ["10M", "100M", "500M"]
TOKENIZER_PATH = Path("tokenizer/tokenizer.json")
RESULTS_PATH = Path("results")
RESULTS_PATH.mkdir(exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
test_tokens = np.memmap(
    "data/test/test_5M.bin",
    dtype=np.uint16,
    mode="r",
    shape=(5_000_000,),
)
tokenizer = Tokenizer.from_file(
    "tokenizer/tokenizer.json"
)
print(tokenizer.get_vocab_size())

results = []

Using device: cuda
16000


In [13]:
@torch.no_grad()
def evaluate_test(
    model,
    tokens,
    batch_size,
    seq_len,
    device,
):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    starts = range(
        0,
        len(tokens) - seq_len - 1,
        seq_len,
    )

    batch_x = []
    batch_y = []

    for start in starts:

        x = tokens[
            start:start + seq_len
        ]

        y = tokens[
            start + 1:start + seq_len + 1
        ]

        batch_x.append(x)
        batch_y.append(y)

        if len(batch_x) == batch_size:

            x = torch.from_numpy(
                np.stack(batch_x).astype(np.int64)
            ).to(device)

            y = torch.from_numpy(
                np.stack(batch_y).astype(np.int64)
            ).to(device)

            outputs = model(
                input_ids=x,
                labels=x,
            )

            loss = outputs.loss

            n = y.numel()

            total_loss += loss.item() * n
            total_tokens += n

            batch_x.clear()
            batch_y.clear()

    if batch_x:
        x = torch.from_numpy(
            np.stack(batch_x).astype(np.int64)
        ).to(device)

        y = torch.from_numpy(
            np.stack(batch_y).astype(np.int64)
        ).to(device)

        outputs = model(
            input_ids=x,
            labels=x,
        )

        loss = outputs.loss

        n = y.numel()

        total_loss += loss.item() * n
        total_tokens += n

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)

    return avg_loss, perplexity

def get_model(model, data):
    return f"{model}_model_on_{data}_tokens.pt"


In [ ]:
for i in range(len(MODELS)):
    for j in range(len(DATA_TOKENS)):

        MODEL_NAME = get_model(MODELS[i], DATA_TOKENS[j])
        MODEL_PATH = MODEL_DIR / MODEL_NAME

        if not MODEL_PATH.exists():
            print(f"Model not found: {MODEL_PATH}")
            continue

        model = create_model(MODELS[i])

        checkpoint = torch.load(
            MODEL_PATH,
            map_location=DEVICE,
        )

        model.load_state_dict(checkpoint["model"])
        model = model.to(DEVICE)
        model.eval()

        test_loss, test_ppl = evaluate_test(
            model=model,
            tokens=test_tokens,
            batch_size=32,
            seq_len=256,
            device=DEVICE,
        )

        for prompt in prompts:
            encoded = tokenizer.encode(prompt)

            input_ids = torch.tensor(
                [encoded.ids],
                dtype=torch.long,
                device=DEVICE,
            )

            with torch.no_grad():
                output_ids = model.generate(
                    input_ids=input_ids,
                    max_new_tokens=100,
                    do_sample=True,
                    temperature=0.8,
                    top_k=50,
                    top_p=0.95,
                    eos_token_id=2,
                    pad_token_id=3,
                )

            generated_text = tokenizer.decode(
                output_ids[0].tolist()
            )

            results.append({
                "model_size": MODELS[i],
                "train_tokens": DATA_TOKENS[j],
                "model_name": MODEL_NAME,
                "checkpoint_step": checkpoint["step"],
                "validation_loss": checkpoint["val_loss"],
                "test_loss": test_loss,
                "test_perplexity": test_ppl,
                "prompt": prompt,
                "generated_text": generated_text,
            })

        del model
        del checkpoint
        del input_ids
        del output_ids
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

Model not found: models\9M_model_on_10M_tokens.pt
Model not found: models\9M_model_on_100M_tokens.pt
Model not found: models\9M_model_on_500M_tokens.pt
Model not found: models\20M_model_on_10M_tokens.pt
Model not found: models\20M_model_on_100M_tokens.pt
Model not found: models\20M_model_on_500M_tokens.pt

Results DataFrame:


,model_size,train_tokens,model_name,checkpoint_step,validation_loss,test_loss,test_perplexity,prompt,generated_text
0,3M,10M,3M_model_on_10M_tokens.pt,1221,6.002743,6.019810,411.500355,Столиця України,"Столиця України та з собою.\nОм:\nКрім того, ..."
1,3M,10M,3M_model_on_10M_tokens.pt,1221,6.002743,6.019810,411.500355,Президент,Президент на 15 квітня «Про це за останні: «П...
2,3M,10M,3M_model_on_10M_tokens.pt,1221,6.002743,6.019810,411.500355,Одного вечора,"Одного вечора, в житті не з'яті, а й в п’юю, ..."
3,3M,10M,3M_model_on_10M_tokens.pt,1221,6.002743,6.019810,411.500355,Найбільша річка в Україні,Найбільша річка в Україні.\nПо-сова система в...
4,3M,10M,3M_model_on_10M_tokens.pt,1221,6.002743,6.019810,411.500355,Ukraine is,Ukraine is the in frodis a ses-ss to wagned f...
5,3M,100M,3M_model_on_100M_tokens.pt,12208,4.514135,4.521850,92.005611,Столиця України,"Столиця України, та її члени (до 20 березня, ..."
6,3M,100M,3M_model_on_100M_tokens.pt,12208,4.514135,4.521850,92.005611,Президент,"Президентов, а також про необхідність проведе..."
7,3M,100M,3M_model_on_100M_tokens.pt,12208,4.514135,4.521850,92.005611,Одного вечора,"Одного вечора, що ви не зможете віднайти, кол..."
8,3M,100M,3M_model_on_100M_tokens.pt,12208,4.514135,4.521850,92.005611,Найбільша річка в Україні,Найбільша річка в Україні є одним з найбільш ...
9,3M,100M,3M_model_on_100M_tokens.pt,12208,4.514135,4.521850,92.005611,Ukraine is,Ukraine is feachespution.\n«Ми вітаємо вас з ...


In [16]:
results_df = pd.DataFrame(results)

summary_df = (
    results_df[
        [
            "model_size",
            "train_tokens",
            "test_loss",
            "test_perplexity",
        ]
    ]
    .drop_duplicates()
    .sort_values(["model_size", "train_tokens"])
    .reset_index(drop=True)
)

summary_df.to_csv(
    RESULTS_PATH / "model_comparison_results.csv",
    index=False,
    encoding="utf-8-sig",
)

prompt_df = results_df[
    [
        "model_size",
        "train_tokens",
        "prompt",
        "generated_text",
    ]
].copy()

prompt_df.to_csv(
    RESULTS_PATH / "model_comparison_prompt_results.csv",
    index=False,
    encoding="utf-8-sig",
)
